# LangChain链式调用(Chain)技术深度解析

## 一、链式调用核心概念

链式调用(Chain)是LangChain框架中最为核心的架构设计思想，它通过将多个功能模块以特定逻辑顺序连接，构建出完整的AI应用处理流水线。这种设计模式类似于工厂中的自动化生产线，每个环节专注于特定任务处理，数据在不同组件间有序流转，最终产出符合预期的结果。

在现代AI应用开发中，链式调用技术解决了几个关键问题：
1. **复杂任务分解**：将综合性AI需求拆解为可管理的子任务单元
2. **处理流程标准化**：建立可重复使用、可监控的任务执行流程
3. **灵活组合能力**：支持根据不同场景动态调整处理逻辑
4. **全流程可观测性**：每个处理环节的输出都可独立验证和调试


### LangChain表达式语言(LCEL)：

LangChain表达式语言（LCEL）是实现链式调用的一种高效方式。LCEL采用声明式方法构建新的可运行组件（Runnables），通过描述应实现的功能而非具体实现方式，使LangChain能够优化链的运行时执行。

在LangChain中，只要实现了Runnable接口并具备invoke方法的组件均可视为链。实现了Runnable接口的类可以接收前一个链的输出作为自身输入。LCEL支持流式输出、异步/并行执行、重试和回退、访问中间结果等。提供了多种链组合方式，使用管道符`|`，该方式不仅书写便捷，且具有强大的表达能力。

LCEL VS LangGraph

LangGraph是LangChain生态中的另一重要工具，它通过图结构解决了传统Agent框架在复杂流程控制上的不足，特别适合需要循环、多角色协作或人工干预的场景，其设计理念在可控性、可靠性和灵活性之间取得了平衡。

LCEL侧重于简化链式调用构建与优化。
LangGraph则专注于复杂流程控制与多角色协作。

### 基础环境配置

首先配置基础环境和DeepSeek客户端：

In [ ]:
from dotenv import load_dotenv, find_dotenv
from langchain_deepseek import ChatDeepSeek

def load_deepseek_config():
    """安全加载DeepSeek API配置
    自动查找并加载项目中的.env文件
    """
    load_dotenv(find_dotenv())

load_deepseek_config() # 初始化环境变量

# 配置DeepSeek聊天模型
llm = ChatDeepSeek(
    model="deepseek-chat", # 指定模型版本
    max_retries=2 # 设置失败重试次数
)

def get_completion(text):
    """简单对话生成函数
    适用于无上下文记忆的单轮对话场景
    """
    messages = [
        ("system", "你是Ai助手，可以帮助我回答生活中的各种问题"),
        ("human", text),
    ]
    response = llm.invoke(messages)
    return response.content


def get_completion_from_messages(messages, temperature=0):
    """带历史上下文的对话生成函数
    Args:
        messages: 完整的消息历史列表
        temperature: 控制生成多样性的参数(0-1)
    """
    llm.temperature = temperature
    response = llm.invoke(messages)
    return response.content

## 二、链式调用类型详解

### 1. 基础链式调用

基础链是最简单的链式结构，通常由"输入-处理-输出"三个基本环节组成。这种链式结构适合处理线性任务流程，例如：
- 用户输入直接传递给提示模板
- 模板输出传递给语言模型
- 模型生成最终响应

这种结构的优势在于实现简单、执行高效，适合处理不需要复杂逻辑转换的标准化任务。

代码示例：

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 创建提示模板
prompt = ChatPromptTemplate.from_template("描述制造{product}的一个公司的最佳名称是什么?")

# 构建简单链：模板 -> LLM
chain = prompt | llm # 使用管道操作符连接组件

# 调用链
print(chain.invoke({"product": "手机"}).content)

### 2. 顺序链(Sequential Chain)

顺序链将多个基础链按固定顺序连接，形成多阶段处理流水线。其核心特点是：
- 前驱链的输出自动成为后继链的输入
- 数据沿单一方向顺序流动
- 适合具有明确阶段划分的任务

典型应用场景包括：
- 多轮信息加工（如：生成→精炼→格式化）
- 分步问题求解（如：分析→计算→验证）
- 递进式内容生成（如：大纲→草稿→终稿）

顺序链通过明确的阶段划分，使得复杂任务的处理过程变得清晰可控。

代码示例：

In [ ]:
# 第一个链：生成公司名称
first_prompt = ChatPromptTemplate.from_template("描述制造{product}的一个公司的最好的名称是什么")
first_chain = first_prompt | llm

# 第二个链：生成公司描述
second_prompt = ChatPromptTemplate.from_template("写一个20字的描述对于下面这个公司，并输出公司名称\
                                                公司：{company_name}")
second_chain = second_prompt | llm

# 组合两个链：第一个链的输出自动作为第二个链的输入
chain = first_chain | second_chain
# 执行完整流程
print(chain.invoke({"product": "手机"}).content)

### 3. 组合链(Composite Chain)

组合链是更为复杂的链式结构，它突破了简单的线性流程，支持：
- **并行执行**：多个链同时处理同一输入的不同方面
- **条件分支**：根据中间结果选择不同处理路径
- **数据聚合**：将多个链的输出合并为统一结果

这种结构通常包含以下组件：
- 路由节点：决定数据处理流向
- 并行处理节点：同时执行多个子任务
- 聚合节点：整合多个处理结果

组合链特别适合处理需要多维度分析、多角度生成的复杂场景。


代码示例：

代码的链式处理流程如下：

1. 输入 `Review` 后进入并行处理阶段，`chain_one` 将 `Review` 翻译成英文，`chain_three` 判断 `Review` 使用的语言，同时保留原始输入。
2. 并行处理的结果进入第一个串行处理阶段，`chain_two` 对英文的 `Review` 进行总结。
3. 第一个串行处理阶段的结果进入第二个串行处理阶段，`chain_four` 根据总结和语言生成后续回复。
4. 最后输出英文的 `Review` `English_Review`、总结`summary`和后续回复`followup_message`。 

```mermaid
graph LR
    classDef startend stroke-width:2px;
    classDef process stroke-width:2px;
    classDef io stroke-width:2px;

    A([输入 Review]):::startend --> B(并行处理):::process
    B --> C(chain_one：Review -> English_Review):::process
    B --> D(chain_three：Review -> language):::process
    C --> F(chain_two：English_Review -> summary):::process
    D --> F(chain_two：English_Review -> summary):::process
    F --> G(chain_four：summary 和 language -> followup_message):::process
    G --> H([输出 English_Review, summary, followup_message]):::startend
```


In [ ]:
from langchain_core.runnables import RunnablePassthrough

# 功能链1 —— 翻译成英语（把下面的review翻译成英语）
# 输入：Review  输出：英文的 Review
first_prompt = ChatPromptTemplate.from_template("把下面的评论review翻译成英文:""\n\n{Review}")
chain_one = {"Review": RunnablePassthrough()} | first_prompt | llm | (lambda x: x.content) # RunnablePassthrough()本质上是一个不做任何处理的数据传递管道，它会将输入数据原封不动地传递给下一个环节。

# 功能链2 —— 用一句话总结下面的 review
# 输入：英文的 Review  输出：Review 的总结
second_prompt = ChatPromptTemplate.from_template("请你用一句话来总结下面的评论review:""\n\n{English_Review}")
chain_two = {"English_Review": lambda x: x["English_Review"]} | second_prompt | llm | (lambda x: x.content)

# 功能链3 —— 判断下面的 Review 使用了什么语言
# 输入：Review  输出：Review 使用的语言
third_prompt = ChatPromptTemplate.from_template("下面的评论review使用的什么语言:\n\n{Review}")
chain_three = {"Review": RunnablePassthrough()} | third_prompt | llm | (lambda x: x.content)

# 功能链4 —— 使用特定的语言对下面的总结写一个后续回复
# 输入：总结，语言  输出： 后续恢复
forth_prompt = ChatPromptTemplate.from_template(
    "使用特定的语言对下面的总结写一个后续回复:""\n\n总结: {summary}\n\n语言: {language}")
chain_four = {"summary": lambda x: x["summary"], "language": lambda x: x["language"]} | forth_prompt | llm \
             | (lambda x: x.content)

# 组合功能链
# 输入 Review   输出：英文Review, 总结， 后续回复
overall_chain = (
        RunnablePassthrough()
        | {
            # 并行执行chain_one和chain_three
            "English_Review": chain_one,
            "language": chain_three,
            "Review": lambda x: x  # 保留原始输入
        }
        | {
            # 串行执行chain_two
            "summary": chain_two,
            "English_Review": lambda x: x["English_Review"],
            "language": lambda x: x["language"]
        }
        | {
            # 串行执行chain_four
            "followup_message": chain_four,
            "English_Review": lambda x: x["English_Review"],
            "summary": lambda x: x["summary"],
            "language": lambda x: x["language"]
        }
)
res = overall_chain.invoke("Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre.J'achète les mêmes dans "
                           "le commerce et le goût est bien meilleur...\nVieux lot oucontrefaçon !?")
print("English_Review:" + res["English_Review"])
print("summary:" + res["summary"])
print("followup_message:" + res["followup_message"])

### 4. 路由链(Router Chain)

路由链是组合链的特化形式，专门用于实现条件分支逻辑。其核心机制包括：
- 输入分类器：分析输入内容特征
- 路由决策：选择最适合的处理分支
- 后备机制：提供默认处理路径

路由链的典型应用场景包括：
- 多领域QA系统（根据问题类型选择专家模型）
- 多语言处理（根据语言特征选择处理管道）
- 多模态输入（根据内容类型选择解析方式）

代码示例：

```mermaid
graph TD
    Input[输入] -->|通过路由链匹配| RouterChain{选择目标链}
    RouterChain -->|物理学| PhysicsChain[物理学链]
    RouterChain -->|数学| MathChain[数学链]
    RouterChain -->|历史| HistoryChain[历史链]
    RouterChain -->|计算机科学| ComputerChain[计算机科学链]
    RouterChain -->|默认| DefaultChain[默认链]
    PhysicsChain --> Output[输出]
    MathChain --> Output
    HistoryChain --> Output
    ComputerChain --> Output
    DefaultChain --> Output
```

In [ ]:
from langchain.chains.router.llm_router import RouterOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain.schema.runnable import RunnableBranch
from langchain_core.runnables import RunnableParallel


# 首先定义不同的提示词模板
# 适用于回答物理问题的提示词模板
physics_template = """你是一个非常聪明的物理专家。 \
你擅长用一种简洁并且易于理解的方式去回答问题。\
当你不知道问题的答案时，你承认\
你不知道.
这是一个问题:
 {input}"""

# 适用于回答数学问题的提示词模板
math_template = """你是一个非常优秀的数学家。 \
你擅长回答数学问题。 \
你之所以如此优秀， \
是因为你能够将棘手的问题分解为组成部分，\
回答组成部分，然后将它们组合在一起，回答更广泛的问题。
这是一个问题：
{input}"""

# 适用于回答历史问题的提示词模板
history_template = """你是以为非常优秀的历史学家。 \
你对一系列历史时期的人物、事件和背景有着极好的学识和理解\
你有能力思考、反思、辩证、讨论和评估过去。\
你尊重历史证据，并有能力利用它来支持你的解释和判断。
这是一个问题:
 {input}"""

# 适用于回答计算机领域问题的提示词模板
computer_template = """ 你是一个成功的计算机科学专家。\
你有创造力、协作精神、\
前瞻性思维、自信、解决问题的能力、\
对理论和算法的理解以及出色的沟通技巧。\
你非常擅长回答编程问题。\
你之所以如此优秀，是因为你知道  \
如何通过以机器可以轻松解释的命令式步骤描述解决方案来解决问题，\
并且你知道如何选择在时间复杂性和空间复杂性之间取得良好平衡的解决方案。
这还是一个输入：
{input}"""

# 对提示模版进行命名和描述
prompt_infos = [
    {
        "name": "物理学",
        "des": "擅长回答关于物理学的问题",
        "template": physics_template
    },
    {
        "name": "数学",
        "des": "擅长回答数学问题",
        "template": math_template
    },
    {
        "name": "历史",
        "des": "擅长回答历史问题",
        "template": history_template
    },
    {
        "name": "计算机科学",
        "des": "擅长回答计算机科学问题",
        "template": computer_template
    }
]

# 基于提示模版信息创建相应目标链
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = prompt | llm
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['des']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

# 创建默认目标链
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = default_prompt | llm

# 路由提示词模板
router_prompt_template = """给语言模型一个原始文本输入，\
让其选择最适合输入的模型提示。\
系统将为您提供可用提示的名称以及最适合该提示的描述。\
如果你认为修改原始输入最终会导致语言模型做出更好的响应，\
你也可以修改原始输入。


<< 格式 >>
返回一个带有JSON对象的markdown代码片段，该JSON对象的格式如下：
```json
{{{{
    "destination": 字符串 \ 使用的提示名字或者使用 "DEFAULT"
    "next_inputs": 字符串 \ 原始输入的改进版本
}}}}

记住：“destination”必须是下面指定的候选提示名称之一，\
或者如果输入不太适合任何候选提示，\
则可以是 “DEFAULT” 。
记住：如果您认为不需要任何修改，\
则 “next_inputs” 可以只是原始输入。
<< 候选提示 >>
    {destinations}
<< 输入 >>
    {{input}}
<< 输出 (记得要包含 ```json)>>
样例:
<< 输入 >>
"什么是黑体辐射?"
<< 输出 >>
```json
{{{{
    "destination": 字符串 \ 使用的提示名字或者使用 "DEFAULT"
    "next_inputs": 字符串 \ 原始输入的改进版本
}}}}
"""

router_template = router_prompt_template.format(destinations=destinations_str)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

# 创建路由链
router_chain = (
        RunnablePassthrough()
        | router_prompt
        | llm
        | JsonOutputParser()  # 解析JSON输出
        | (lambda x: x["destination"])
)

# 创建整体链路
# 运用 RunnableBranch 分支选择器，按顺序检查每个条件，执行第一个满足条件的分支，若都不满足则执行默认分支。
chain = {
            "input": lambda x: x,
            "dest": router_chain  # 先获取路由结果
        } \
        | RunnableBranch(
    *[
        (lambda x, name=name: x["dest"] == name, chain) for name, chain in destination_chains.items()
    ],
    default_chain
) \
        | (lambda x: x.content)

# 创建带回调的链(便于查看执行过程中到底执行了哪条链)
chain_with_log = RunnableParallel({
    "input": lambda x: x,
    "dest": router_chain  # 保留完整的路由信息
}) | {
    "response": RunnableBranch(
        *[(lambda x, name=name: x["dest"] == name, chain)
          for name, chain in destination_chains.items()],
        default_chain
    ),
    "selected_route": lambda x: x["dest"]  # 显式保留路由选择
} | RunnableParallel({
    "final_response": lambda x: x["response"].content,
    "route_used": lambda x: x["selected_route"]
})

res = chain_with_log.invoke("TCP和UDP的区别")
print(f"回答内容：{res['final_response']}")
print(f"使用的路径：{res['route_used']}")

链式调用技术正在成为构建复杂AI系统的标准范式，通过灵活组合各种处理单元，开发者可以构建出适应各种业务场景的智能应用。随着技术的不断发展，链式调用将进一步提升其表达能力、执行效率和易用性，成为AI工程化的重要基石。

链式调用技术正在成为构建复杂AI系统的标准范式，通过灵活组合各种处理单元，开发者可以构建出适应各种业务场景的智能应用。随着技术的不断发展，链式调用将进一步提升其表达能力、执行效率和易用性，成为AI工程化的重要基石。